In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
import numpy as np

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")


Using mps device


# Experimentation

In [ ]:
W = torch.tensor([[3,-2]])
W.shape

torch.Size([1, 2])

In [ ]:
b = torch.tensor([1])

In [ ]:
X = torch.arange(6).reshape(3,2)

X@W.T + b

tensor([[-1],
        [ 1],
        [ 3]])

In [ ]:
class Exercise_Two(nn.Module):

    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(2,3, bias = True)
    
        with torch.no_grad():
            self.layer_1.weight.copy_(torch.tensor([[1,0],
                                                   [0,1],
                                                   [-1,-1]]))
            self.layer_1.bias.copy_(torch.tensor([0.,0.,1.]))
        
        for p in self.parameaters():
            p.requires_grad_(False)
    
    def forward(self, x):
        return self.layer_1(x)

## Problem 10 in Sample Problems

In [ ]:
class My_ReLU(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self,x):
        return torch.clamp(x, min = 0)


In [ ]:
model = My_ReLU().to(device)
print(model)

My_ReLU()


In [ ]:
x = torch.tensor([-2.0, 0.0, 3.5])
print(model(x))  # tensor([0.0000, 0.0000, 3.5000])


tensor([0.0000, 0.0000, 3.5000])


## Problem 11

In [ ]:
import torch
import torch.nn as nn

class TriangleInside(nn.Module):
    """
    1 if (x,y) is inside/on triangle with vertices (0,0),(1,0),(0,2), else 0.
    Uses fixed Linear layers + torch.where (no separate Step class).
    """
    def __init__(self):
        super().__init__()

        self.halfspaces = nn.Linear(2, 3, bias=True)
        self.and_layer  = nn.Linear(3, 1, bias=True)

        #being more careful with no_grad()
        with torch.no_grad():
            # Half-space tests: ax + by + c >= 0
            self.halfspaces.weight.copy_(torch.tensor([
                [ 1.0,  0.0],   # x >= 0
                [ 0.0,  1.0],   # y >= 0
                [-2.0, -1.0],   # -2x - y + 2 >= 0
            ]))

            self.halfspaces.bias.copy_(torch.tensor([0.0, 0.0, 2.0]))

            # AND: h1+h2+h3 >= 3  <=>  (sum - 2.5) >= 0
            self.and_layer.weight.copy_(torch.tensor([[1.0, 1.0, 1.0]]))
            self.and_layer.bias.copy_(torch.tensor([-2.5]))

        # for p in self.parameters():
        #     p.requires_grad_(False)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        h = self.halfspaces(X)                          # (B,3)
        h = torch.where(h >= 0, 1.0, 0.0)              # Theta, (B,3)
        out = self.and_layer(h)                         # (B,1)
        out = torch.where(out >= 0, 1.0, 0.0)          # Theta, (B,1)
        return out

# quick check
m = TriangleInside()
pts = torch.tensor([[0.1,0.1],[0.6,1.0],[-0.1,0.5],[0.0,2.0]], dtype=torch.float32)
print(m(pts))


tensor([[1.],
        [0.],
        [0.],
        [1.]])


# Problem 12

In [ ]:
from torchvision.models import resnet34
import torchsummary
model = resnet34(pretrained=True)

/Users/eric/code/GitHub/USAAIO/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/eric/code/GitHub/USAAIO/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /Users/eric/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100.0%


## 12.1

In [ ]:
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(total)

21797672


In [ ]:
help(torch.numel)

Help on built-in function numel in module torch:

numel(...)
    numel(input: Tensor) -> int
    
    Returns the total number of elements in the :attr:`input` tensor.
    
    Args:
        input (Tensor): the input tensor.
    
    Example::
    
        >>> a = torch.randn(1, 2, 3, 4, 5)
        >>> torch.numel(a)
        120
        >>> a = torch.zeros(4,4)
        >>> torch.numel(a)
        16



## 12.2

In [ ]:
for p in model.parameters():
    print(p.shape)

torch.Size([64, 3, 7, 7])
torch.Size([64])
torch.Size([64])
torch.Size([64, 64, 3, 3])
torch.Size([64])
torch.Size([64])
torch.Size([64, 64, 3, 3])
torch.Size([64])
torch.Size([64])
torch.Size([64, 64, 3, 3])
torch.Size([64])
torch.Size([64])
torch.Size([64, 64, 3, 3])
torch.Size([64])
torch.Size([64])
torch.Size([64, 64, 3, 3])
torch.Size([64])
torch.Size([64])
torch.Size([64, 64, 3, 3])
torch.Size([64])
torch.Size([64])
torch.Size([128, 64, 3, 3])
torch.Size([128])
torch.Size([128])
torch.Size([128, 128, 3, 3])
torch.Size([128])
torch.Size([128])
torch.Size([128, 64, 1, 1])
torch.Size([128])
torch.Size([128])
torch.Size([128, 128, 3, 3])
torch.Size([128])
torch.Size([128])
torch.Size([128, 128, 3, 3])
torch.Size([128])
torch.Size([128])
torch.Size([128, 128, 3, 3])
torch.Size([128])
torch.Size([128])
torch.Size([128, 128, 3, 3])
torch.Size([128])
torch.Size([128])
torch.Size([128, 128, 3, 3])
torch.Size([128])
torch.Size([128])
torch.Size([128, 128, 3, 3])
torch.Size([128])
torch.Siz

In [ ]:
print(model.layer3)

Sequential(
  (0): BasicBlock(
    (conv1): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (downsample): Sequential(
      (0): Conv2d(128, 256, kernel_size=(1, 1), stride=(2, 2), bias=False)
      (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (1): BasicBlock(
    (conv1): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(256, eps=1

In [ ]:
model.layer2

Sequential(
  (0): BasicBlock(
    (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (downsample): Sequential(
      (0): Conv2d(64, 128, kernel_size=(1, 1), stride=(2, 2), bias=False)
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (1): BasicBlock(
    (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(128, eps=1e-

In [ ]:
help(nn.Conv2d)

Help on class Conv2d in module torch.nn.modules.conv:

class Conv2d(_ConvNd)
 |  Conv2d(in_channels: int, out_channels: int, kernel_size: Union[int, tuple[int, int]], stride: Union[int, tuple[int, int]] = 1, padding: Union[str, int, tuple[int, int]] = 0, dilation: Union[int, tuple[int, int]] = 1, groups: int = 1, bias: bool = True, padding_mode: Literal['zeros', 'reflect', 'replicate', 'circular'] = 'zeros', device=None, dtype=None) -> None
 |  
 |  Applies a 2D convolution over an input signal composed of several input
 |  planes.
 |  
 |  In the simplest case, the output value of the layer with input size
 |  :math:`(N, C_{\text{in}}, H, W)` and output :math:`(N, C_{\text{out}}, H_{\text{out}}, W_{\text{out}})`
 |  can be precisely described as:
 |  
 |  .. math::
 |      \text{out}(N_i, C_{\text{out}_j}) = \text{bias}(C_{\text{out}_j}) +
 |      \sum_{k = 0}^{C_{\text{in}} - 1} \text{weight}(C_{\text{out}_j}, k) \star \text{input}(N_i, k)
 |  
 |  
 |  where :math:`\star` is the val

In [ ]:
model.layer2[0]

BasicBlock(
  (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (downsample): Sequential(
    (0): Conv2d(64, 128, kernel_size=(1, 1), stride=(2, 2), bias=False)
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
)

In [ ]:
backbone = resnet34(weights=True)
for p in backbone.parameters():
    p.requires_grad = False



# 3) Replace the final classification layer (fc) for 12 labels
in_features = backbone.fc.in_features  # number of features coming into fc (for resnet34, this is 512)
backbone.fc = nn.Linear(in_features, 12)  # new head is learnable by default

model = backbone

/Users/eric/code/GitHub/USAAIO/.venv/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Frozen:", frozen)
print("Trainable:", trainable)


Frozen: 21284672
Trainable: 6156


In [ ]:
class MLP(nn.module):
    def __init__(self):
        self.flatten = nn.Flatten()
        self.sequential = nn.Sequential(
            nn.Linear(20,30),
            nn.ReLU(),
            nn.Linear(30,30),
            nn.Sigmoid()
        )
    def forward(self):
        

In [ ]:
help(nn.Sequential)

NameError: name 'nn' is not defined